# 02 — Deep Dive: Ragas Metrics

**Track:** Intermediate · **Stage:** Evaluation

RAG systems fail in two distinct ways:
1. **Retrieval Failure:** The database returned the wrong documents.
2. **Generation Failure:** The database returned the right documents, but the LLM ignored them, hallucinated, or didn't answer the question.

If you only evaluate the final answer, you don't know *what* to fix. Frameworks like **Ragas** split evaluation into 4 core metrics using an **LLM-as-a-Judge**.

## 1. The Ragas Metrics Framework

- **Context Precision (Retrieval):** Are the relevant documents ranked at the top?
- **Context Recall (Retrieval):** Did we retrieve all the necessary information to answer the ground truth question?
- **Faithfulness (Generation):** Is the generated answer entirely supported by the retrieved context (no hallucinations)?
- **Answer Relevance (Generation):** Does the generated answer actually address the user's question?

In [ ]:
# !pip install langchain langchain-core

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.fake import FakeListLLM
from langchain_core.output_parsers import StrOutputParser

## 2. Faithfulness (No Hallucinations)

An answer is faithful if all claims made in the answer can be inferred from the given context.

In [ ]:
faithfulness_template = """
You are an impartial judge. 
Read the Generated Answer and the Retrieved Context.
Are ALL claims in the Generated Answer directly supported by the Retrieved Context?
Output strictly '1' for Yes, '0' for No.

Retrieved Context: {context}
Generated Answer: {answer}
"""
faithfulness_prompt = ChatPromptTemplate.from_template(faithfulness_template)

# Scenario: The LLM hallucinated a specific exception not found in the text.
context = "PolicyAssist Auto: Comprehensive auto insurance does not cover mechanical failure."
bad_answer = "Comprehensive auto insurance does not cover mechanical failure, unless it occurs within the first 30 days."

judge_llm = FakeListLLM(responses=["0"])
faithfulness_chain = faithfulness_prompt | judge_llm | StrOutputParser()

print("--- Evaluating Faithfulness ---")
score = faithfulness_chain.invoke({"context": context, "answer": bad_answer})
print(f"Answer: {bad_answer}")
print(f"Faithfulness Score: {score} (Failed due to hallucinated 30-day rule)")

## 3. Answer Relevance

An answer can be perfectly faithful to the text, but completely fail to answer the user's question.

In [ ]:
relevance_template = """
You are an impartial judge.
Does the Generated Answer directly address the User Question?
Output strictly '1' for Yes, '0' for No.

User Question: {question}
Generated Answer: {answer}
"""
relevance_prompt = ChatPromptTemplate.from_template(relevance_template)

# Scenario: The user asks about water damage, but the retriever pulled auto policies, so the LLM talked about cars.
question = "What is the cap for residential water damage coverage?"
unhelpful_answer = "Comprehensive auto insurance does not cover mechanical failure."

judge_llm = FakeListLLM(responses=["0"])
relevance_chain = relevance_prompt | judge_llm | StrOutputParser()

print("\n--- Evaluating Answer Relevance ---")
score = relevance_chain.invoke({"question": question, "answer": unhelpful_answer})
print(f"Question: {question}")
print(f"Answer: {unhelpful_answer}")
print(f"Answer Relevance Score: {score} (Failed due to topic mismatch)")

## 4. Context Recall

Did we retrieve the necessary information? We check if the Ground Truth Answer can be found in the Retrieved Context.

In [ ]:
recall_template = """
You are an impartial judge.
Can the Ground Truth Answer be entirely deduced from the Retrieved Context?
Output strictly '1' for Yes, '0' for No.

Ground Truth Answer: {ground_truth}
Retrieved Context: {context}
"""
recall_prompt = ChatPromptTemplate.from_template(recall_template)

# Scenario: The retriever missed the document about the FR-99 rider.
ground_truth = "$50,000 unless the FR-99 rider is purchased."
bad_context = "Water damage coverage is capped at $50,000."

judge_llm = FakeListLLM(responses=["0"])
recall_chain = recall_prompt | judge_llm | StrOutputParser()

print("\n--- Evaluating Context Recall ---")
score = recall_chain.invoke({"ground_truth": ground_truth, "context": bad_context})
print(f"Ground Truth: {ground_truth}")
print(f"Context: {bad_context}")
print(f"Context Recall Score: {score} (Failed due to missing the rider information)")

## Reflection

1. **Automation:** In practice, you do not write these prompts manually. You install the `ragas` library and run `ragas.evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])`. Ragas handles the LLM calls and complex mathematical aggregations automatically.
2. **The LLM Judge:** These metrics rely entirely on the judging LLM being smart and impartial. Always use the most capable model available (e.g., GPT-4o or Claude 3.5 Sonnet) as your judge, even if you use a smaller, cheaper model (like GPT-4o-mini) for the actual RAG generation.